In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 36. B7 — 時系列・状態空間・動的金利モデル

> B7の対象は「日次」と呼ばれるカレンダー等間隔系列ではなく、Treasuryの公表観測日で進む曲線系列である。

## 学習目標

- stationarity、forecast origin、horizonをデータ契約として書ける
- AR/VAR、Kalman filter、Dynamic Nelson–Siegel、GARCHの役割を分けられる
- filtered estimateとsmoothed estimateの情報集合を区別できる
- B5/B6の外部テストを変更せず5公表日先の曲線予測を評価できる
- 統計的予測精度と取引可能な経済価値を区別できる

## 前提知識

- B1のleast squares、PCA、Nelson–Siegel
- B2のMarkov過程と条件付き期待値
- B3の時系列依存を考慮した推論
- B5/B6のpoint-in-time splitとlocked outer test

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 36


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. B7のevidence chain

| Week | Core | Treasury lab | 主な反証 |
|---|---|---|---|
| 25 | stationarity、AR、forecast evaluation | 10年CMTのlevel/change | random walkに勝たない |
| 26 | VAR、Granger、IRF、cointegration | NS factor dynamics | predictive contentをcausalityと誤読 |
| 27 | Kalman filter/smoother、missing data | Dynamic Nelson–Siegel | smoother leakage |
| 28 | GARCH、break、regime dependence | 10年変化のconditional variance | volatility proxyをrealized volatilityと呼ぶ |

Primary horizonは5 Treasury publication observations。1と20はsecondaryで、カレンダー日へ読み替えない。

In [4]:
factor_panel = qt.extract_nelson_siegel_factors(curve_yields, maturity_years, 0.5)
factor_changes = np.diff(factor_panel, axis=0) * 100.0
summary = pd.DataFrame(
    {
        "factor": ["level", "slope", "curvature"],
        "mean_change_bp": factor_changes.mean(axis=0),
        "standard_deviation_bp": factor_changes.std(axis=0, ddof=1),
        "lag1_autocorrelation": [qt.autocorrelation(factor_changes[:, i], 1)[1] for i in range(3)],
    }
)
display(summary)

fig = go.Figure()
for index, name in enumerate(["level", "slope", "curvature"]):
    fig.add_scatter(x=change_dates, y=factor_changes[:, index], name=name, mode="lines")
fig.add_vline(x=pd.Timestamp(test_start_date).timestamp() * 1000, line_dash="dash", line_color="black")
fig.update_layout(
    title="Fixed-decay Nelson-Siegel factor changes and locked test boundary",
    xaxis_title="Treasury publication date",
    yaxis_title="Factor change (bp)",
    template="plotly_white",
)
fig.show()

,factor,mean_change_bp,standard_deviation_bp,lag1_autocorrelation
0,level,0.077506,5.447817,-0.016950
1,slope,0.064720,6.151040,-0.007117
2,curvature,-0.075054,17.250616,-0.057008


## 2. Project contract

Targetは5公表観測先の5 tenor曲線。forecast originではその日までの公表値だけを使う。B5/B6のtest開始日は固定し、B7で後ろへずらしたりmodel selectionへ再利用したりしない。公式CMTはpar yieldの公表系列であり、取引価格、zero curve、intraday quoteではない。

## 3. 失敗モード

- 不規則な休場間隔をカレンダー日等間隔と呼ぶ
- levelの高い自己相関を予測改善と混同する
- full sampleで次数、decay、state数を選ぶ
- smootherをforecast originの特徴量にする
- 公式公表yieldだけからPnLやhedge実現値を作る

## 4. 段階別演習

### 基礎

1. publication horizonとcalendar horizonの差を三連休の例で説明せよ。
2. level、change、factor changeのACFを比較せよ。

### 標準

3. validationだけでAR次数を選ぶprotocolを書け。
4. missing tenorを持つ日をKalman updateがどう扱うか式で示せ。

### 研究

5. 2007–2025拡張manifestを作る場合の構造変化auditを事前登録せよ。

## 5. Exit Criteria

- [ ] publication observationを時間単位として明記した
- [ ] primary 5、secondary 1/20のhorizonを固定した
- [ ] filteredとsmoothedの情報集合を区別した
- [ ] B5/B6 outer testを再利用し再選択に使わない
- [ ] pricing、PnL、causalityのunsupported claimを除外した

## 6. 出典


- [Forecasting: Principles and Practice — Stationarity and differencing](https://otexts.com/fpp3/stationarity.html)
- [Forecasting: Principles and Practice — ARIMA models](https://otexts.com/fpp3/arima.html)
- [U.S. Treasury Yield Curve Methodology](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/treasury-yield-curve-methodology)

- [Kalman (1960), A New Approach to Linear Filtering and Prediction Problems](https://people.math.harvard.edu/archive/116_fall_03/handouts/Kalman1960.pdf)
- [Särkkä and Svensson, Bayesian Filtering and Smoothing, 2nd ed.](https://users.aalto.fi/~ssarkka/pub/bfs_book_2023_online.pdf)
- [Diebold and Li, Forecasting the Term Structure of Government Bond Yields](https://www.nber.org/papers/w10048.pdf)